# lenia-vlm on Colab

Runs the Lenia simulator on a GPU (PyTorch) with results saved to your Google Drive.

**Before the first run**
1. *Runtime → Change runtime type → GPU* (a T4 is fine).
2. *Secrets* (the key icon in the left sidebar): add `GITHUB_TOKEN`, a fine-grained GitHub token with
   read-only *Contents* access to `feli10/lenia-vlm`, and turn on notebook access. Only needed while
   the repo is private.
3. Optional, only for VLM judging: add `OPENROUTER_API_KEY` and `TYPESAFE_API_KEY` the same way.
   They are written to a `.env` on this temporary machine and never printed.

Run the cells top to bottom. Re-running the setup cells is harmless (they pull the latest code).

## 1. GPU and Drive

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime -> Change runtime type -> GPU"
import os
from google.colab import drive
drive.mount('/content/drive')
RUNS = '/content/drive/MyDrive/lenia-runs'      # every result goes here and survives disconnects
os.makedirs(RUNS, exist_ok=True)
print('results folder:', RUNS)

## 2. Code, dependencies, data

In [ ]:
import os, subprocess
from google.colab import userdata

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

REPO, DIR = 'github.com/feli10/lenia-vlm.git', '/content/lenia-vlm'
tok = secret('GITHUB_TOKEN')
url = f'https://x-access-token:{tok}@{REPO}' if tok else f'https://{REPO}'
if not os.path.isdir(DIR):
    r = subprocess.run(['git', 'clone', '-q', url, DIR], capture_output=True, text=True)
else:
    r = subprocess.run(['git', '-C', DIR, 'pull', '-q', url, 'main'], capture_output=True, text=True)
print('git:', 'ok' if r.returncode == 0 else r.stderr.replace(tok or '\0', '***'))
subprocess.run(['git', '-C', DIR, 'remote', 'set-url', 'origin', f'https://{REPO}'])   # don't keep the token in .git
os.chdir(DIR)
!git log --oneline -1
!pip install -q -e .

In [ ]:
# Chan's catalogue (MIT licence), and API keys for the judges if you added them as secrets
import os
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/animals.json'):
    !curl -sL -o data/animals.json https://raw.githubusercontent.com/Chakazul/Lenia/master/Python/animals.json
print('catalogue:', os.path.getsize('data/animals.json'), 'bytes')
keys = {k: secret(k) for k in ('OPENROUTER_API_KEY', 'TYPESAFE_API_KEY')}
with open('.env', 'w') as f:
    for k, v in keys.items():
        if v:
            f.write(f'{k}={v}\n')
print('judge keys present:', [k for k, v in keys.items() if v] or 'none (fine for simulation-only runs)')

## 3. Check the GPU simulator, then time it

Compares the PyTorch step with the numpy reference from identical starts, then times both. Do not
use the GPU for runs unless this ends with **ALL CHECKS PASSED**.

In [ ]:
!python -m lenia --backend auto bench

Optional: the full test suite (about a minute).

In [ ]:
!pip install -q pytest && python -m pytest -q tests

## 4. A first GPU run

A small multikernel survey at double resolution, straight to Drive. Compare its time with the same
command on your Mac to see the real end-to-end speedup (the bench above times the update step only;
the alive classifier still runs on the CPU).

In [ ]:
import time
t0 = time.time()
!python -m lenia --backend auto --runs {RUNS} --config configs/r26.yaml survey --mode multikernel --n 64 --out {RUNS}/gpu-test-survey
print(f'{time.time() - t0:.0f} s')

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f'{RUNS}/gpu-test-survey/sheet_alive.png'))[:1]:
    display(Image(p, width=700))
!cat {RUNS}/gpu-test-survey/summary.json | head -20

## 5. Experiment A: multi-channel creatures at higher resolution

Starts from Chan's `found/` collections (about 2,600 creatures with several kernels and channels), at
their own kernel radius (8-41) in a 256x256 world.

1. **Download** the collections (small; re-run after a disconnect, the machine is wiped).
2. **Survey**: run each creature from its own pattern and keep the living ones. `--n-per-file 100`
   takes a sample (about 800 creatures, roughly 15-30 minutes on a T4); drop it for all ~2,600.
3. **Search**: novelty search from 64 diverse living creatures, with the same hard checks as before
   (alive at 100, still alive at 200, coherent at 2x resolution). It saves a checkpoint to Drive every
   2 minutes: after a disconnect, re-run sections 1-2 and this cell, and it continues. To extend a
   finished run, raise `--evals` and re-run.

In [ ]:
!python -m lenia found download

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/r26.yaml found survey --n-per-file 100 --out {RUNS}/found-survey

In [ ]:
from IPython.display import Image, display
import glob, json
print(json.dumps(json.load(open(f'{RUNS}/found-survey/summary.json'))['per_file'], indent=1)[:1500])
for p in sorted(glob.glob(f'{RUNS}/found-survey/sheet_*.png'))[:4]:
    display(Image(p, width=700))

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/r26.yaml explore --from-found {RUNS}/found-survey --roots 64 --evals 3200 --out {RUNS}/expA-explore

In [ ]:
!python -m lenia --backend auto --runs {RUNS} --config configs/r26.yaml explore --replot {RUNS}/expA-explore --no-catalogue
for name in ('sheet_novel.png', 'sheet_lineages.png'):
    display(Image(f'{RUNS}/expA-explore/{name}', width=700))

## Notes

- Every command takes `--backend auto --runs {RUNS}`; results then live in your Drive folder.
- The free tier disconnects idle sessions and caps sessions at about 12 hours. `lenia explore` and
  `lenia guided run` resume after a disconnect (re-run sections 1-2, then the same command with the
  same `--out`). After a disconnect the machine is wiped, your Drive is not.
- Claude can read your Drive folder: just say when a run has finished.